# SBi-FAGAT v3 — Cloud GPU Benchmark Runner (Google Colab T4 / Kaggle GPU)

Offloads all remaining multi-seed GNN benchmarks (VN Temporal Holdout, VN Ablation & $K$-Curve, and optional Track 2 PHEME Tier 2A $K=10, N \ge 15$) from local CPU to **Google Colab T4** or **Kaggle T4/P100 GPU** in ~3-5 minutes.

### Quick Start
1. **Colab:** Select `Runtime -> Change runtime type -> T4 GPU`. Upload your local `data/processed/vn/` (`nodes.parquet`, `edges.parquet`, `nlp_features.pt`, and optional `data/processed/pheme/`) to Google Drive (e.g., `MyDrive/SBiFAGAT/data/processed/vn`), or mount Drive in Cell 2.
2. **Kaggle:** Enable `Accelerator -> GPU T4` and attach your processed dataset (`nodes.parquet`, `edges.parquet`, `nlp_features.pt`) under `/kaggle/input/...`.
3. Run all cells top-to-bottom. At the end, download `sbifagat_v3_gpu_results.zip` and extract into your local repo root before assembling the paper.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hieu-kiien/gnn-nlp-propagation.git"
REPO_BRANCH = "main"

is_kaggle = Path("/kaggle/working").exists()
work_root = Path("/kaggle/working/gnn-nlp-propagation") if is_kaggle else Path("/content/gnn-nlp-propagation")

if not Path("scripts/run_experiments.py").exists():
    if work_root.exists():
        os.chdir(work_root)
        subprocess.run(["git", "fetch", "--all"], check=True)
        subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(work_root)], check=True)
        os.chdir(work_root)

subprocess.run(["nvidia-smi"], check=False)
try:
    import torch_geometric  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"], check=True)
if Path("requirements.txt").exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)

import torch
print("Working dir:", Path.cwd())
print("PyTorch:", torch.__version__, "CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))


In [ ]:
import shutil
from pathlib import Path

# Adjust GDRIVE_DATA_ROOT if your processed datasets live elsewhere in Drive
GDRIVE_DATA_ROOT = Path("/content/drive/MyDrive/SBiFAGAT/data/processed")
IN_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except ImportError:
    pass

REQUIRED_FILES = ("nodes.parquet", "edges.parquet", "nlp_features.pt")


def _stage_corpus(corpus: str) -> bool:
    dst_dir = Path("data/processed") / corpus
    dst_dir.mkdir(parents=True, exist_ok=True)
    if all((dst_dir / f).exists() for f in REQUIRED_FILES):
        print(f"[{corpus}] Already staged at {dst_dir}")
        return True

    search_roots = [GDRIVE_DATA_ROOT / corpus, Path("/kaggle/input")]
    found: dict[str, Path] = {}
    for root in search_roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file() or p.name not in REQUIRED_FILES:
                continue
            parent_hint = p.parent.name.lower()
            if corpus in parent_hint or root == (GDRIVE_DATA_ROOT / corpus):
                found.setdefault(p.name, p)
    if all(f in found for f in REQUIRED_FILES):
        for fname, src_p in found.items():
            tgt_p = dst_dir / fname
            if not tgt_p.exists():
                try:
                    tgt_p.symlink_to(src_p)
                except OSError:
                    shutil.copy2(src_p, tgt_p)
        print(f"[{corpus}] Linked/staged from {found['nlp_features.pt'].parent}")
        return True

    print(f"[{corpus}] Not found locally; copy {REQUIRED_FILES} into {dst_dir.resolve()} if running this track.")
    return False


vn_ready = _stage_corpus("vn")
pheme_ready = _stage_corpus("pheme")
print("Dataset availability -> VN:", vn_ready, "| PHEME:", pheme_ready)


In [ ]:
import subprocess
import sys
from pathlib import Path

# Set True if you also want to re-produce checkpoints/best_sbifagat_vn.pt and main VN figures on GPU
RERUN_VN_CANONICAL = False


def _run(mod: str, *args: str) -> None:
    cmd = [sys.executable, "-u", "-m", mod, *args]
    print(">>>", " ".join(cmd))
    subprocess.run(cmd, check=True)


if Path("data/processed/vn/nlp_features.pt").exists():
    if RERUN_VN_CANONICAL:
        _run(
            "scripts.run_experiments",
            "--data-dir", "data/processed/vn",
            "--k-horizon", "20",
            "--hidden-dim", "64",
            "--edge-dim", "2",
            "--use-cascade-stats",
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
            "--epochs", "100",
            "--patience", "10",
        )

    _run(
        "scripts.run_experiments",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--split-mode", "temporal",
        "--edge-source", "teacher",
        "--epochs", "100",
        "--patience", "10",
        "--skip-figures",
    )

    _run(
        "scripts.run_ablation",
        "--data-dir", "data/processed/vn",
        "--k-horizon", "20",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "25",
        "--polar-target-mode", "future_only",
        "--epochs", "100",
        "--patience", "10",
    )

    vn_ckpt = Path("checkpoints/best_sbifagat_vn.pt")
    if vn_ckpt.exists():
        _run(
            "scripts.analyze_mechanisms",
            "--data-dir", "data/processed/vn",
            "--ckpt", str(vn_ckpt),
            "--k-horizon", "20",
            "--min-total-replies", "25",
            "--polar-target-mode", "future_only",
        )
else:
    print("Skipping VN benchmarks: data/processed/vn/nlp_features.pt not found.")


In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("data/processed/pheme/nlp_features.pt").exists():
    subprocess.run([
        sys.executable, "-u", "-m", "scripts.export_dataset_stats",
        "--data-dir", "data/processed/pheme",
        "--k-horizon", "10",
        "--min-total-replies", "15",
        "--polar-target-mode", "future_only",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "-m", "scripts.run_experiments",
        "--data-dir", "data/processed/pheme",
        "--k-horizon", "10",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "15",
        "--polar-target-mode", "future_only",
        "--epochs", "100",
        "--patience", "10",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "-m", "scripts.run_experiments",
        "--data-dir", "data/processed/pheme",
        "--k-horizon", "10",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "15",
        "--polar-target-mode", "future_only",
        "--eval-protocol", "loo",
        "--edge-source", "teacher",
        "--epochs", "100",
        "--patience", "10",
        "--skip-figures",
    ], check=True)
    subprocess.run([
        sys.executable, "-u", "-m", "scripts.run_ablation",
        "--data-dir", "data/processed/pheme",
        "--k-horizon", "10",
        "--hidden-dim", "64",
        "--edge-dim", "2",
        "--use-cascade-stats",
        "--min-total-replies", "15",
        "--polar-target-mode", "future_only",
        "--epochs", "100",
        "--patience", "10",
    ], check=True)
    pheme_ckpt = Path("checkpoints/best_sbifagat_pheme.pt")
    if pheme_ckpt.exists():
        subprocess.run([
            sys.executable, "-u", "-m", "scripts.analyze_mechanisms",
            "--data-dir", "data/processed/pheme",
            "--ckpt", str(pheme_ckpt),
            "--k-horizon", "10",
            "--min-total-replies", "15",
            "--polar-target-mode", "future_only",
        ], check=True)
else:
    print("Skipping optional Track 2 PHEME Tier 2A: data/processed/pheme/nlp_features.pt not found.")


In [ ]:
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run([
    sys.executable, "-u", "-m", "scripts.assemble_paper_artifacts",
    "--paper-dir", "paper",
], check=True)

BUNDLE_NAME = Path("sbifagat_v3_gpu_results.zip")
FORBIDDEN_SUBSTRS = ("_machine_key", ".secret", ".env")
FORBIDDEN_EXTS = {".parquet", ".zip", ".bak"}

added = []
with zipfile.ZipFile(BUNDLE_NAME, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root_name in ("paper", "checkpoints"):
        root_dir = Path(root_name)
        if not root_dir.exists():
            continue
        for p in sorted(root_dir.rglob("*")):
            if not p.is_file():
                continue
            low = str(p).lower()
            if any(s in low for s in FORBIDDEN_SUBSTRS) or p.suffix.lower() in FORBIDDEN_EXTS:
                continue
            if p.name == "nlp_features.pt":
                continue
            zf.write(p, arcname=p.as_posix())
            added.append(p.as_posix())

print(f"Packaged {len(added)} result files into {BUNDLE_NAME.resolve()}")
try:
    from google.colab import files  # type: ignore
    files.download(str(BUNDLE_NAME))
except ImportError:
    pass
